Recovered from the kernel's run history (session of 2026-09-30 16:29): the cells exactly as executed, in execution order.

In [1]:
import regex as re

In [2]:
def get_stats(ids)-> dict: # this is just 
    counts = {}
    for i in range(len(ids)):
        for pair in zip(ids[i], ids[i][1:]):
            counts[pair] = counts.get(pair, 0) + 1
    return counts

In [3]:
def merge(ids, pair, idx):

    out = [[] for _ in range (len(ids))]
    for n in range(len(ids)):
        ids_n = ids[n]
        i = 0
        while i < len(ids_n):
            if i < len(ids_n) - 1 and ids_n[i] == pair[0] and ids_n[i + 1] == pair[1]:
                out[n].append(idx)
                i += 2
            else:
                out[n].append(ids_n[i])
                i += 1
    return out

In [4]:
pattern = re.compile("|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""\p{N}{1,3}""",
    r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""",
    r"""\s*[\r\n]+""",
    r"""\s+(?!\S)""",
    r"""\s+""",
]))

In [5]:
pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [6]:
# ai genertaed 

def get_earliest_merge(token_list, merges):
    """
    Finds the adjacent pair in token_list that has the earliest merge rank
    (lowest integer key) in the merges dictionary.
    
    merges format: {rank_int: [item0, item1]}
    """
    # 1. Invert the dict so we can look up ranks in O(1): (item0, item1) -> rank
    pair_to_rank = {tuple(pair): rank for rank, pair in merges.items()}

    # 2. Extract unique adjacent pairs present in the list
    if token_list and isinstance(token_list[0], (list, tuple)):
        # If the list is already given as candidate pairs
        pairs_in_list = {tuple(p) for p in token_list}
    else:
        # If the list is a sequence of tokens/bytes
        pairs_in_list = {
            (token_list[i], token_list[i + 1])
            for i in range(len(token_list) - 1)
        }

    # 3. Filter for pairs that exist in merges
    valid_candidates = [p for p in pairs_in_list if p in pair_to_rank]

    if not valid_candidates:
        return None, None

    # 4. Pick the pair with the lowest rank (merged first)
    best_pair = min(valid_candidates, key=lambda p: pair_to_rank[p])
    best_rank = pair_to_rank[best_pair]

    return best_rank, list(best_pair)

In [7]:
import re

data = ["apple,banana;orange", "carrot|pea,bean", "dog;cat|bird"]
delimiters = [",", ";", "|"]

# Build a safe regex pattern from your list: ",|;|\|"
pattern = "|".join(map(re.escape, delimiters))

nested_result = [re.split(pattern, s) for s in data]

print(nested_result)

[['apple', 'banana', 'orange'], ['carrot', 'pea', 'bean'], ['dog', 'cat', 'bird']]


In [8]:
def train(text, vocab_size, special_tokens,verbose=False): 

    # pieces =re.split(re.escape(special_tokens), text) 


    remove_st = "|".join(map(re.escape, special_tokens))

    pieces = re.split(remove_st, text) 
 
    print(f"pieces : {pieces}")
    pre_tokens = [] # This just have no more of the special token 
    for piece in pieces:
        pre_tokens.extend(pattern.findall(piece)) # this create a list of new word

    print(f"pretokens: {pre_tokens}")

    ids = list(chunk.encode("utf-8") for chunk in pre_tokens)

    vocab = {i: bytes([i]) for i in range(256)}
 
    print(f"ids {ids}") 
    merges_list = []
    merges = {} # (97,98) -> 257 

    for idx in range(256, vocab_size - len(special_tokens)): # 256 -> 398 
        stats = get_stats(ids)
        if not stats:                                # nothing left to merge
            break
        pair = max(stats, key=lambda p: (stats[p], vocab[p[0]], vocab[p[1]]), )        # tie breaking base on the bytes if want to go ofr the lower use - ? 
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        merges_list.append((vocab[pair[0]], vocab[pair[1]]))
        if verbose:
            print(f"merge {idx - 255}/{vocab_size - 256}: {pair} -> {idx} {vocab[idx]} ({stats[pair]}x)")
 
    for i in range(len(special_tokens)):

        vocab[vocab_size-i-1] = special_tokens[i].encode("utf-8")
 
    return vocab, merges_list

In [9]:
def encode(text):

    pieces = re.split(re.escape("<|endoftext|>"), text)

    target_value = "<|endoftext|>"

    key = next((k for k, v in vocab.items() if v == target_value), None)

    # another layer to actually use the regex pattenr using extends cuz 

    pre_tokens = [] 

    for piece in pieces:
        pre_tokens.extend(pattern.findall(piece))

    ids =  [token.encode("utf-8") for token in pre_tokens]
    # dont forget that token is a nested loop so we need to loop inside it 
    for idx in range(len(ids)):
        stats = get_stats(pre_tokens)   

        # idx is not a [] it's a number 
        i_ids = ids[idx] # ids is just a list of byes 

        for i_idx in i_ids:
            for i in range(256, len(merges)):
                if (merges[i][0] == i_ids[i_idx] and merges[i+1] == i_ids[i_idx+1]): # might have some out of loop error
                    pair = merges[i]
                    ids = merge(ids, pair, merges[i])
                 
    return ids 


In [19]:
special_token = "<|endoftext|>"

with open("../data/small.txt") as f:
    text = f.read()

vocab, merges = train(text, 400, special_token)

pieces : ['u ', '', '', "'", ' hav', ' ', '', ' b', ' scar', '', ' ', '', ' ', 'h', ' l', 'u', ' ', '', "g, I'll pr", '', '', 'c', ' y', 'u". Th', ' m', 'l', ' ', '', 'l', ' s', ' sa', '', ' wi', 'h ', 'h', ' li', '', 'l', ' girl. Sh', ' was v', 'ry ki', '', ' a', '', ' ', 'h', ' m', 'l', ' s', '', '', ' cam', ' ', '', ' ', 'rus', ' h', 'r. H', ' l', 'a', '', '', ' agai', 's', ' h', 'r a', '', ' sh', ' k', 'p', ' him sa', '', '. Th', ' m', 'l', ' ha', ' ', '', 'u', '', ' his b', 's', ' ', 'ri', '', '', '.\n', '', '', '', '', '', '', '', '', '', '', '', '', '\nO', 'c', ' up', '', ' a ', 'im', ', i', ' a warm a', '', ' su', '', 'y plac', ', ', 'h', 'r', ' was a big pi', '. A li', '', 'l', ' b', 'y ', 'am', '', ' T', 'm lik', '', ' ', '', ' play ', '', 'ar ', 'h', ' pi', '. O', '', ' ', 'ay, T', 'm l', 's', ' his r', '', ' ball. H', ' was v', 'ry sa', '.\nT', 'm ask', '', ' his ', 'ri', '', '', ', Sam, ', '', ' h', 'lp him s', 'arch ', '', 'r ', 'h', ' ball. Th', 'y l', '', 'k', '', ' hig

AttributeError: 'str' object has no attribute 'findall'

In [11]:
def encode(text):

    # encode from text to remove the special = [ ]
    special_token = "<|endoftext|>"

    pattern = f"({re.escape(special_token)})"
    chunks = re.split(pattern, text)

    # remove the end of text ? 
    pre_tokens = [] 

    for chunk in chunks:
        if chunk in vocab:
            pre_tokens.append() 

    for i in range(len(chunks)) :
        if isinstance(chunks[i], str): # eot is also type string 
            pre_tokens.extend(pattern.findall(chunks[i]))
        else:
            pre_tokens[i] = special_token


    # keys,value = next((k, v) for k, v in vocab.items() if v == special_token)

    
    # for pre_token in pre_tokens:
    #     if(pre_token == value):
    #         pre_token = keys


    print(pre_tokens)

    ids = [
    token.encode("utf-8") if isinstance(token, str) else token # conversion only works when it is type not byte 
    for token in pre_tokens
    ]


    new_ids = []

    for token_index in range(len(ids)):  

        token_bytes = ids[token_index]  

        if len(token_bytes) >= 2 : 
            while True :   

                chunk, best_rank = get_earliest_merge(token_bytes,merges) 

                if best_rank == None  or len(chunk) ==0:
                   break 

                else:
                    pair = merges[best_rank] 
                    token_bytes = merge(chunk, pair, best_rank)  
            new_ids.extend(token_bytes)
        else :
            new_ids.extend(token_bytes)        
                 
    return new_ids 


In [14]:
def decode(ids):
    pass

In [13]:
enc = encode("Hello <|endoftext|> world")

NameError: name 'vocab' is not defined